In [ ]:
!sudo apt install osmctools

In [ ]:
!pip install networkx[default]

In [ ]:
!pip install pyrosm geopandas shapely matplotlib osmnx

In [ ]:
import osmnx as ox
from pyrosm import OSM
import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import random
import numpy as np
import datetime
import time
from tqdm import tqdm
import math
from sklearn.metrics.pairwise import haversine_distances
import numpy as np
import heapq
import copy
import json

In [ ]:
def shapely_polygon_to_polyfile(polygon, filename="hochiminh.poly", name="TP_HCM"):
    """
    Lưu shapely Polygon hoặc MultiPolygon thành file .poly theo định dạng OSM
    """
    def write_ring(f, coords):
        for x, y in coords:
            f.write(f"  {x} {y}\n")
        f.write("END\n")

    with open(filename, "w") as f:
        f.write(f"{name}\n")
        if polygon.geom_type == "Polygon":
            f.write("1\n")
            write_ring(f, polygon.exterior.coords)
            f.write("END\n")
        elif polygon.geom_type == "MultiPolygon":
            for i, poly in enumerate(polygon.geoms, 1):
                f.write(f"{i}\n")
                write_ring(f, poly.exterior.coords)
            f.write("END\n")
        else:
            raise ValueError("Not a valid Polygon or MultiPolygon")

In [ ]:
import osmnx as ox
from shapely.geometry import Polygon, MultiPolygon

# Lấy polygon TP.HCM
gdf = ox.geocode_to_gdf("Ho Chi Minh City, Vietnam")
geom = gdf.geometry.values[0]  # shapely Polygon or MultiPolygon

# Tạo file .poly
shapely_polygon_to_polyfile(geom, "hochiminh.poly")

In [ ]:
!osmconvert "/kaggle/input/vietnammap/vietnam-latest.osm.pbf" -B=hochiminh.poly -o=hochiminh.osm.pbf

In [ ]:
osm = OSM("hochiminh.osm.pbf")

In [ ]:
nodes, edges = osm.get_network(network_type="all", nodes=True)

In [ ]:
def haversine_distance(G, u, v):
    node_u = G.nodes[u]
    ux = node_u['x']
    uy = node_u['y']

    node_v = G.nodes[v]
    vx = node_v['x']
    vy = node_v['y']

    point_u = np.radians([[uy, ux]])
    point_v = np.radians([[vy, vx]])
    
    distance = haversine_distances(point_u, point_v)[0][0] * 6371000
    return distance

In [ ]:
import pandas as pd
import geopandas as gpd
def expand_edges_by_oneway(edges: pd.DataFrame) -> pd.DataFrame:
    new_edges = []

    for idx, row in edges.iterrows():
        u, v = row['u'], row['v']
        length = row.get('length')
        if length == None or u == v:
            continue
        oneway = str(row.get('oneway')).lower()
        row_data = row.to_dict()

        if oneway == 'yes':
            new_edges.append({**row_data, 'u': u, 'v': v})
        elif oneway == '-1':
            new_edges.append({**row_data, 'u': v, 'v': u})
        elif oneway in ['no', 'none', 'nan']:
            # thêm cả hai chiều
            new_edges.append({**row_data, 'u': u, 'v': v})
            new_edges.append({**row_data, 'u': v, 'v': u})
        else:
            # coi mặc định là hai chiều nếu nhãn lạ
            new_edges.append({**row_data, 'u': u, 'v': v})
            new_edges.append({**row_data, 'u': v, 'v': u})
    new_edges = pd.DataFrame(new_edges)
    edges_clean = gpd.GeoDataFrame(new_edges, geometry=new_edges['geometry'], crs=edges.crs)
    
    return edges_clean

def preprocess_graph(G):
    id_map = {nid: i for i, nid in enumerate(G.nodes)}
    G = nx.relabel_nodes(G, id_map, copy=True)
    
    edges_to_remove = []

    for u, v, k, data in G.edges(keys=True, data=True):
        if data.get('length') is None:
            edges_to_remove.append((u, v, k))

    G.remove_edges_from(edges_to_remove)

    edges_to_add = []
    edges_to_remove = []

    for u, v, k, data in G.edges(keys=True, data=True):
        oneway = str(data.get('oneway')).lower()
        if u == v:
            edges_to_remove.append((u, v, k))
            continue
        
        if oneway == 'yes':
            continue
        elif oneway == '-1':
            edges_to_add.append((v, u, data.copy()))
            edges_to_remove.append((u, v, k))
        elif oneway in ['no', 'none', 'nan']:
            edges_to_add.append((v, u, data.copy()))
        else:
            edges_to_add.append((v, u, data.copy()))

    for u, v, k in edges_to_remove:
        G.remove_edge(u, v, key=k)
    
    for u, v, data in edges_to_add:
        G.add_edge(u, v, **data)

    return G

In [ ]:
print(len(nodes))
print(len(edges))

In [ ]:
graph = osm.to_graph(nodes, edges, graph_type='networkx')

In [ ]:
graph = preprocess_graph(graph)

In [ ]:
print(type(graph))

In [ ]:
bidirectional_edges = set()
unidirectional_edges = set()

for u, v, k in graph.edges(keys=True):
    if graph.has_edge(v, u):  # nếu có cạnh ngược lại
        # Sắp xếp lại cặp cho thống nhất (A, B) == (B, A)
        bidirectional_edges.add(tuple(sorted((u, v))))
    else:
        unidirectional_edges.add((u, v))

# Loại bỏ những cặp đã được xem là bidirectional khỏi unidirectional
# unidirectional_edges = {e for e in unidirectional_edges if tuple(sorted(e)) not in bidirectional_edges}

print("Cạnh hai chiều:", len(bidirectional_edges))
print("Cạnh một chiều:", len(unidirectional_edges))

In [ ]:
print(len(graph.nodes))
print(len(graph.edges))

In [ ]:
N = len(nodes)  # số điểm muốn chọn
sample_nodes = nodes.sample(n=N, random_state=42)

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(sample_nodes["lon"], sample_nodes["lat"], c="red", s=40)

plt.title("Một vài điểm ngẫu nhiên từ TP.HCM")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)
plt.show()

In [ ]:
print(edges.keys())

In [ ]:
print(edges['u'])

In [ ]:
print(len(graph.edges))

In [ ]:
def shortest_path_alogrithm(G, source, target, algo='dijkstra', heuristic='haversine'):
    t0, t1 = None, None
    if algo == 'dijkstra':
        t0 = time.time()
        path = nx.dijkstra_path(G, source, target, weight='length')
        t1 = time.time()
    elif algo == 'astar':
        if heuristic == 'haversine':
            heuristic = haversine_distance
        t0 = time.time()
        path = nx.astar_path(G, source, target, heuristic=lambda u, v: heuristic(G, u, v), weight='length')
        t1 = time.time()
    else:
        raise Exception("No algo in database")
    
    return {
        "algorithm": algo,
        "source": source,
        "target": target,
        "latency_s": round((t1 - t0), 4),
        "path_length": nx.path_weight(G, path, weight='length'),
        "path_nodes": path
    }

In [ ]:
def stratified_random_queries(G, k=1000, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    degree_dict = dict(G.degree())
    nodes = list(G.nodes())
    
    low_deg = [n for n in nodes if degree_dict[n] <= 4]
    med_deg = [n for n in nodes if 5 <= degree_dict[n] <= 9]
    high_deg = [n for n in nodes if degree_dict[n] >= 10]

    queries = []
    k_per_group = k // 3

    def sample_group(group, num_samples):
        samples = []
        group = list(set(group))
        for _ in range(num_samples):
            if len(group) < 2:
                break
            u, v = random.sample(group, 2)
            samples.append((u, v))
        return samples

    queries.extend(sample_group(low_deg, k_per_group))
    queries.extend(sample_group(med_deg, k_per_group))
    queries.extend(sample_group(high_deg, k - len(queries)))

    return queries


In [ ]:
def run_shortest_path_benchmark(G, benchmark, algo='dijkstra', heuristic='haversine'):
    log = []
    for u, v in tqdm(benchmark):
        log.append(shortest_path_alogrithm(G, u, v, algo, heuristic))
    
    now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")  
    output_file = algo + "_" + now + ".json"
    with open(output_file, 'w') as f:
        json.dump(log, f, indent=4)

In [ ]:
queries = stratified_random_queries(graph)

In [ ]:
for node in graph.nodes:
    if graph.nodes.get(node, 'y') == None:
        print(node)

In [ ]:
run_shortest_path_benchmark(graph, queries, algo='dijkstra')

In [ ]:
len(queries)

In [ ]:
queries[:5]